# Task5 — Rank fusion 95/5

รวม Density auto 95% กับ Conditional quantile residual 5% โดยใช้ percentile ranks และเลือก top 2,046 บัญชี คง Task1–4 จาก Density auto submission ที่ผู้ใช้รายงานคะแนนรวม 0.81771

Run All คำนวณ fusion ใหม่จาก saved scores แล้วสร้าง CSV เมื่อไฟล์ยังไม่มี ถ้ามีแล้วจะตรวจว่าได้คำตอบเดิมโดยไม่ overwrite ไม่มีการฝึกโมเดลหรืออัปโหลด Kaggle

## 1. Setup

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Task5/models/rank_fusion_followup.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within CPE342-Kaggle")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from Task5.models import rank_fusion_followup as fusion
print("Source:", fusion.SOURCE)
print("Output:", fusion.OUTPUT)

Source: /Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/submission/kaggle_t12345_t1_slow800_bagging_v4_t3_conditional_quantile_v1_t4_resnet18_cosine18_tta_v1_t5_density_auto_v1.csv
Output: /Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/submission/kaggle_t12345_t1_slow800_bagging_v4_t3_conditional_quantile_v1_t4_resnet18_cosine18_tta_v1_t5_rank_fusion95_5_v1.csv


## 2. Input validation

ตรวจ IDs, row order, finite scores และ Task1–4 ของ reference ทั้งสองตรงกัน รวมถึง proof ว่า quantile models ผ่านการ reload ตรวจสอบแล้ว

In [2]:
density_scores, residual_scores, columns, source_rows, baseline, family_reference = fusion.load_inputs()
display(pd.DataFrame({"input": ["Density auto", "Conditional quantile residual", "Family cap2 reference"], "rows": [len(density_scores), len(residual_scores), len(family_reference)], "alerts": [int(baseline.sum()), None, int(family_reference.sum())]}))

,input,rows,alerts
0,Density auto,25889,2046.0
1,Conditional quantile residual,25889,NaN
2,Family cap2 reference,25889,6531.0


## 3. Calculate percentile ranks and fusion

`fusion_score = 0.95 × rank_average(density_score)/N + 0.05 × rank_average(conditional_residual_score)/N`

ใช้คะแนน original ranking_score จาก density และ conditional_residual_score จาก quantile models ไม่ใช้ binary predictions มารวมกัน ถ้า fused scores เท่ากันจะใช้ original test-row order เป็นตัวตัดสิน

In [3]:
scores, sensitivity, summary = fusion.build()
display(scores[["test_row_index", "player_id", "density_percentile", "residual_percentile", "fusion_score", "baseline_task5", "task5"]].head(10))

,test_row_index,player_id,density_percentile,residual_percentile,fusion_score,baseline_task5,task5
0,0,P024743,0.939163,0.333578,0.908884,1,1
1,1,P005319,0.523775,0.316351,0.513403,0,0
2,2,P014970,0.984743,0.162270,0.943619,1,1
3,3,P003275,0.660435,0.240218,0.639424,0,0
4,4,P009022,0.397466,0.751671,0.415176,0,0
5,5,P012665,0.228669,0.902507,0.262360,0,0
6,6,P022542,0.136931,0.353355,0.147752,0,0
7,7,P005299,0.832168,0.922863,0.836703,0,0
8,8,P009096,0.631426,0.800997,0.639905,0,0
9,9,P005284,0.121828,0.772954,0.154384,0,0


## 4. Weight sensitivity

ตารางนี้วัดการเปลี่ยนคำตอบ ไม่ได้วัด F3 หรือเลือกน้ำหนักจาก labels เลือก 95/5 หลังพิจารณา sensitivity โดยไม่มี measured anomaly validation เลือกจำนวน alerts เท่า baseline เพื่อแยก ranking ออกจาก operating budget

In [4]:
display(sensitivity[["density_weight", "residual_weight", "alerts", "retained_density_alerts", "added", "removed", "jaccard_density", "jaccard_family_cap2"]])

,density_weight,residual_weight,alerts,retained_density_alerts,added,removed,jaccard_density,jaccard_family_cap2
0,1.000,0.000,2046,2046,0,0,1.000000,0.237662
1,0.990,0.010,2046,2012,34,34,0.967308,0.235879
2,0.975,0.025,2046,1965,81,81,0.923836,0.234989
3,0.950,0.050,2046,1889,157,157,0.857467,0.234989
4,0.925,0.075,2046,1841,205,205,0.817859,0.232682
5,0.900,0.100,2046,1797,249,249,0.783007,0.230913
6,0.850,0.150,2046,1681,365,365,0.697221,0.221447
7,0.800,0.200,2046,1592,454,454,0.636800,0.214873
8,0.700,0.300,2046,1489,557,557,0.572032,0.206159
9,0.500,0.500,2046,1385,661,661,0.511636,0.199580


## 5. Selected 95/5 result

In [5]:
display(pd.DataFrame([summary]))
changed = scores[scores.baseline_task5.ne(scores.task5)].copy()
display(changed[["id", "baseline_task5", "task5", "density_percentile", "residual_percentile", "fusion_score"]].head(15))
print("Changed rows:", len(changed))

,rows,alerts,retained_density_alerts,added,removed,changed_task5_rows,jaccard_density,jaccard_family_cap2,source_public_score_reported,candidate_public_score,anomaly_labels_available,anomaly_f3
0,25889,2046,1889,157,157,314,0.857467,0.234989,0.81771,None,False,None


,id,baseline_task5,task5,density_percentile,residual_percentile,fusion_score
43,ANS00044,0,1,0.918614,0.868554,0.916111
75,ANS00076,0,1,0.906910,0.869558,0.905043
181,ANS00182,0,1,0.909228,0.950095,0.911271
206,ANS00207,1,0,0.921511,0.036734,0.877272
576,ANS00577,0,1,0.908224,0.838387,0.904732
606,ANS00607,0,1,0.911236,0.777859,0.904568
656,ANS00657,1,0,0.924138,0.402101,0.898036
736,ANS00737,1,0,0.923442,0.375874,0.896064
749,ANS00750,0,1,0.911739,0.951524,0.913728
1019,ANS01020,0,1,0.912395,0.794700,0.906510


Changed rows: 314


## 6. Export submission CSV

เปลี่ยนเฉพาะ task5 ใน source CSV เก็บ id และ Task1–4 เป็น literal CSV fields เดิม สร้างไฟล์ใหม่ใน submission/ ไม่แก้ไฟล์คะแนนเดิม ถ้ามี output แล้วจะตรวจแทนการ overwrite การสร้างครั้งแรกใช้ local Artifact export runtime; การตรวจ output ที่มีแล้วใช้ Python ได้

In [6]:
export_check = fusion.export_submission()
print(fusion.OUTPUT)
display(pd.read_csv(fusion.OUTPUT).head())

/Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/submission/kaggle_t12345_t1_slow800_bagging_v4_t3_conditional_quantile_v1_t4_resnet18_cosine18_tta_v1_t5_rank_fusion95_5_v1.csv


,id,task1,task2,task3,task4,task5
0,ANS00001,1,3,0.000000,0,1
1,ANS00002,1,2,2332.195747,2,0
2,ANS00003,1,2,0.000000,3,1
3,ANS00004,1,1,0.000000,0,0
4,ANS00005,0,0,0.000000,3,0


## 7. Independent verification

อ่าน exported CSV ด้วย Python csv ตรวจทุกแถว และคำนวณ ranks/top-k ซ้ำด้วย stable argsort ตรวจว่ามี 25,889 แถว, 2,046 alerts และเปลี่ยนเฉพาะ Task5 จำนวน 314 แถว

In [7]:
verification = fusion.verify_csv()
verification

{'rows': 25889,
 'columns': ['id', 'task1', 'task2', 'task3', 'task4', 'task5'],
 'ids_and_tasks1_to4_literal_fields_preserved': True,
 'task5_predictions_independently_recomputed': True,
 'task5_alerts': 2046,
 'changed_task5_rows': 314,
 'output_sha256': '4e3878630a9d3d02ce14d03824c99295cf1e5e5d4a96132a0294728aaf10a34a',
 'measured_anomaly_f3': None}

## 8. Interpretation

95/5 รักษา alerts ของ density 1,889 บัญชี และสลับเข้า/ออกอย่างละ 157 บัญชี ความใกล้ baseline ไม่ยืนยันว่าคะแนนสูงขึ้น ไม่มี measured Task5 F3 หรือ Kaggle score ของไฟล์ใหม่นี้ จำนวน 2,046 alerts เป็น experimental policy ที่ยืมจาก baseline ไม่ใช่ true anomaly prevalence

Artifacts และ source hashes อยู่ใน models/results/rank_fusion95_5_v1/ และมี JSON provenance คู่กับ submission CSV